# Scraping Data Ulasan Aplikasi Gojek (Google Play Store)

**Proyek Analisis Sentimen — Dicoding**

Notebook ini melakukan **scraping mandiri** terhadap ulasan aplikasi **Gojek** (`com.gojek.app`) dari Google Play Store menggunakan pustaka [`google-play-scraper`](https://pypi.org/project/google-play-scraper/).

- **Sumber data:** Google Play Store (ulasan pengguna), bahasa Indonesia (`lang='id'`, `country='id'`).
- **Target jumlah:** ±15.000 ulasan mentah, agar setelah pembersihan & deduplikasi tetap tersisa **≥ 10.000** sampel.
- **Output:** `data/gojek_reviews_raw.csv`.

> Catatan etika: data diambil melalui API publik `google-play-scraper`, tanpa kredensial/login, dan hanya berupa ulasan publik. Sesuai dengan pedoman Dicoding mengenai scraping yang beretika.

In [1]:
# Instalasi pustaka (jalankan jika belum ter-install)
# !pip install google-play-scraper pandas

import time
import pandas as pd
from google_play_scraper import reviews, Sort

print('Pustaka berhasil di-import.')

Pustaka berhasil di-import.


## 1. Konfigurasi & Fungsi Scraping

Kita mengambil ulasan secara bertahap (paginasi) menggunakan `continuation_token`. Setiap panggilan mengambil hingga 200 ulasan, lalu token dipakai untuk mengambil batch berikutnya hingga target terpenuhi.

In [2]:
APP_ID = 'com.gojek.app'   # ID aplikasi Gojek di Google Play Store
TARGET = 15000            # target jumlah ulasan mentah
BATCH = 200               # jumlah ulasan per permintaan


def scrape_reviews(app_id, target, batch_size=200, sleep=0.4):
    """Mengambil ulasan secara bertahap memakai continuation_token."""
    collected = []
    token = None
    batch_no = 0
    while len(collected) < target:
        result, token = reviews(
            app_id,
            lang='id',
            country='id',
            sort=Sort.NEWEST,
            count=batch_size,
            continuation_token=token,
        )
        if not result:
            print('Tidak ada data tambahan, berhenti.')
            break
        collected.extend(result)
        batch_no += 1
        if batch_no % 5 == 0 or len(collected) >= target:
            print(f'Batch {batch_no:>3} | total terkumpul: {len(collected)}')
        if token is None:
            print('continuation_token habis, berhenti.')
            break
        time.sleep(sleep)  # jeda sopan agar tidak membebani server
    return collected

In [3]:
raw_reviews = scrape_reviews(APP_ID, TARGET, BATCH)
print(f'\nTotal ulasan mentah terkumpul: {len(raw_reviews)}')

Batch   5 | total terkumpul: 1000


Batch  10 | total terkumpul: 2000


Batch  15 | total terkumpul: 3000


Batch  20 | total terkumpul: 4000


Batch  25 | total terkumpul: 5000


...

Batch  60 | total terkumpul: 12000


Batch  65 | total terkumpul: 13000


Batch  70 | total terkumpul: 14000


Batch  75 | total terkumpul: 15000



Total ulasan mentah terkumpul: 15000


## 2. Susun ke DataFrame & Pembersihan Awal

Kita ambil kolom yang relevan, lalu buang duplikat dan ulasan yang terlalu pendek/kosong.

In [4]:
df = pd.DataFrame(raw_reviews)[['reviewId', 'userName', 'content', 'score', 'thumbsUpCount', 'at']]
print('Ukuran sebelum pembersihan:', df.shape)
df.head()

Ukuran sebelum pembersihan: (15000, 6)


,reviewId,userName,content,score,thumbsUpCount,at
0,5dded7fd-34f6-4415-940d-375fa4a02684,Muhammad Arif rahman,bagus,5,0,2026-05-23 02:27:12
1,541e18cf-db50-4e55-971a-8438de5a34f8,Finan Tio Azkaban,mantap,5,0,2026-05-22 23:59:08
2,5dca9b77-8886-4888-ad32-14b806e14dc2,Gita Mishael1910,Good,5,0,2026-05-22 23:55:46
3,1593424f-7d4c-4732-85a0-497b80fc1cd7,David Zakaria,eeewwww di t,4,0,2026-05-22 23:55:28
4,ffb17dd5-73eb-4007-bcc7-a51faeec123e,Khori Bahtirar,"susah cari driver, banyak kendaraan driver yan...",1,0,2026-05-22 23:12:24


In [5]:
# Buang baris tanpa konten, hapus duplikat berdasarkan teks & reviewId, buang ulasan terlalu pendek
df = df.dropna(subset=['content'])
df['content'] = df['content'].astype(str).str.strip()
df = df[df['content'].str.len() >= 5]
df = df.drop_duplicates(subset=['reviewId'])
df = df.drop_duplicates(subset=['content'])
df = df.reset_index(drop=True)

print('Ukuran setelah pembersihan:', df.shape)
assert len(df) >= 10000, f'Jumlah data ({len(df)}) kurang dari 10.000! Tambah TARGET dan ulangi scraping.'
print('OK: jumlah sampel memenuhi syarat (>= 10.000).')

Ukuran setelah pembersihan: (10928, 6)
OK: jumlah sampel memenuhi syarat (>= 10.000).


In [6]:
# Distribusi rating bintang (informasi awal, label sentimen dibuat di notebook pelatihan)
print('Distribusi skor rating:')
print(df['score'].value_counts().sort_index())

Distribusi skor rating:
score
1    4161
2     737
3     603
4     505
5    4922
Name: count, dtype: int64


## 3. Simpan Dataset Mentah

In [7]:
import os
os.makedirs('data', exist_ok=True)
OUT_PATH = 'data/gojek_reviews_raw.csv'
df.to_csv(OUT_PATH, index=False)
print(f'Dataset mentah disimpan ke: {OUT_PATH}')
print(f'Jumlah baris: {len(df)} | Jumlah kolom: {df.shape[1]}')

Dataset mentah disimpan ke: data/gojek_reviews_raw.csv
Jumlah baris: 10928 | Jumlah kolom: 6
